In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from notebookutils import mssparkutils

print("Attached lakehouses:", [lh.name for lh in mssparkutils.lakehouse.listLakehouses()])
print("Listing Files/Bronze ...")
for f in mssparkutils.fs.ls("Files/Bronze"):
    print(f.path, f.size)

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 3, Finished, Available, Finished, False)

AttributeError: module 'notebookutils.mssparkutils.lakehouse' has no attribute 'listLakehouses'

In [2]:
# === CONFIG ===
RAW_DATASET_PATH = "Files/Bronze/Swiggy_Food_Delivery_Dataset.csv"
DATA_DICTIONARY_PATH = "Files/Bronze/Swiggy_Orders_Data_Dictionary.csv"

RAW_TABLE = "bronze_food_orders"      # managed Delta table to create
RAW_VIEW  = "vw_bronze_food_orders"   # temp view for ad-hoc SQL

# === IMPORTS ===
from pyspark.sql import functions as F, types as T

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 4, Finished, Available, Finished, False)

In [3]:
raw_df = (spark.read
    .option("header", True)
    .option("inferSchema", True)      # ok at Bronze stage
    .option("multiLine", True)        # supports rows spanning multiple lines
    .option("escape", '"')            # handles commas inside quoted text
    .csv(RAW_DATASET_PATH)
    .withColumn("load_ts", F.current_timestamp())
)

print("Raw row count:", raw_df.count())
raw_df.printSchema()
display(raw_df.limit(20))

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 5, Finished, Available, Finished, False)

Raw row count: 4000
root
 |-- order_id: string (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- delivery_time: timestamp (nullable = true)
 |-- restaurant: string (nullable = true)
 |-- cuisine: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- rating: string (nullable = true)
 |-- location: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ordered_qty: integer (nullable = true)
 |-- book_table: string (nullable = true)
 |-- online_order: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- load_ts: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, c4a65974-4f81-48ed-90f0-b486a418e01d)

In [4]:
try:
    dict_df = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(DATA_DICTIONARY_PATH))
    display(dict_df)
except Exception as e:
    print("Data dictionary read skipped/error:", e)

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 6, Finished, Available, Finished, False)

Data dictionary read skipped/error: [PATH_NOT_FOUND] Path does not exist: abfss://a7508e07-63db-4890-849b-76c27162ca04@onelake.dfs.fabric.microsoft.com/dc2c4045-593b-4dfe-9c6e-815dc3995db9/Files/Bronze/Swiggy_Orders_Data_Dictionary.csv.


In [6]:
# Overwrite if you are re-running
(raw_df
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(RAW_TABLE)
)

# Also register a temp view for quick SQL in this notebook session
#spark.catalog.dropTempView(RAW_VIEW) if RAW_VIEW in [v.name for v in spark.catalog.listTempViews()] else None
#raw_df.createOrReplaceTempView(RAW_VIEW)

print(f"Saved Bronze Delta table: {RAW_TABLE}")


StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 8, Finished, Available, Finished, False)

Saved Bronze Delta table: bronze_food_orders


In [7]:
spark.sql(f"SELECT COUNT(*) AS rows FROM {RAW_TABLE}").show()
display(spark.table(RAW_TABLE).limit(10))

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 9, Finished, Available, Finished, False)

+----+
|rows|
+----+
|4000|
+----+



SynapseWidget(Synapse.DataFrame, 529d1c0d-0582-4f0e-9a9b-4dbb961aff08)

In [8]:
# ===== CONFIG =====
RAW_DATASET_PATH = "Files/Bronze/Swiggy_Food_Delivery_Dataset.csv"  # adjust if your file name differs
SILVER_TABLE     = "silver_food_orders"                             # final cleaned Delta table
SILVER_VIEW      = "vw_silver_food_orders"                          # temp view for ad-hoc SQL

# ===== IMPORTS =====
from pyspark.sql import functions as F, types as T

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 10, Finished, Available, Finished, False)

In [9]:
# Compact whitespace + trim + Title Case
def norm(col):
    return F.initcap(F.regexp_replace(F.trim(F.col(col)), r"\s+", " "))

# Normalize Yes/No/Unknown for raw flags
def yn_unknown(col):
    c = F.lower(F.trim(F.col(col)))
    return (F.when(F.col(col).isNull(), F.lit("Unknown"))
             .when(c.isin("yes","y","true","1"), F.lit("Yes"))
             .when(c.isin("no","n","false","0"), F.lit("No"))
             .otherwise(F.lit("Unknown")))

# Canonical boolean from a Yes/No/Unknown-like column
def to_bool_from_yn(col):
    c = F.lower(F.trim(F.col(col)))
    return F.when(c.isin("yes","y","true","1"), F.lit(True)).otherwise(F.lit(False))

# Rating parser: "4.5/5" -> 4.5 ; "NEW"/"NA"/"" -> NULL
def parse_rating(col):
    txt = F.upper(F.trim(F.col(col)))
    num = F.regexp_extract(F.col(col), r"(\d+(\.\d+)?)", 1).cast("double")
    return F.when(txt.isin("NEW","NA",""), None).otherwise(num)

# Fix Bengaluru locality variants (extend as needed when you see your data)
def fix_location(col):
    loc = F.lower(F.trim(F.col(col)))
    fixed = (F.when(loc == "koramangla", "Koramangala")
              .when(loc == "kormangala", "Koramangala")
              .when(loc == "white field", "Whitefield")
              .when(loc == "marathalli", "Marathahalli")
              .when(loc == "b.t.m layout", "BTM Layout")
              .when(loc == "b t m layout", "BTM Layout")
              .when(loc == "indira nagar", "Indiranagar")
              .otherwise(F.initcap(F.regexp_replace(F.col(col), r"\s+", " "))))
    return fixed

# Ensure we have a city; fix Bangalore variants → Bengaluru
def fix_city(col):
    c = F.lower(F.trim(F.col(col)))
    return (F.when(c.rlike(r"\b(bnegaluru|bangaluru|bengalur|bangalore)\b"), "Bengaluru")
             .otherwise(F.initcap(F.col(col))))

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 11, Finished, Available, Finished, False)

In [10]:
df = raw_df

# --- Strings ---
if "restaurant" in df.columns:
    df = df.withColumn("restaurant", norm("restaurant"))
if "cuisine" in df.columns:
    df = df.withColumn("cuisine", norm("cuisine"))
if "location" in df.columns:
    df = df.withColumn("location", fix_location("location"))

# City (if absent, create one since dataset is Bengaluru-focused)
if "city" in df.columns:
    df = df.withColumn("city", fix_city("city"))
else:
    df = df.withColumn("city", F.lit("Bengaluru"))

# --- Timestamps ---
for tcol in ["order_time", "delivery_time"]:
    if tcol in df.columns:
        df = df.withColumn(tcol, F.to_timestamp(F.col(tcol)))  # adjust format if needed

# --- Ratings ---
if "rating" in df.columns:
    df = df.withColumn("rating_num",
           F.when(parse_rating("rating") < 0, 0.0)
            .when(parse_rating("rating") > 5, 5.0)
            .otherwise(parse_rating("rating"))
    )

# --- Amount ---
if "total_amount" in df.columns:
    df = df.withColumn("total_amount", F.col("total_amount").cast("double"))
    neg_cnt = df.filter(F.col("total_amount") < 0).count()
    print(f"Rows with negative total_amount (will be removed): {neg_cnt}")
    df = df.filter((F.col("total_amount").isNull()) | (F.col("total_amount") >= 0))

# --- Distance ---
if "distance_km" in df.columns:
    df = df.withColumn("distance_km",
        F.when(F.col("distance_km").cast("double") <= 0, None)
         .otherwise(F.col("distance_km").cast("double"))
    )

# --- Categorical imputation (Yes/No/Unknown) and booleans ---
if "online_order" in df.columns:
    df = df.withColumn("online_order_cat", yn_unknown("online_order"))
    df = df.withColumn("online_order", to_bool_from_yn("online_order_cat"))
else:
    df = df.withColumn("online_order_cat", F.lit("Unknown"))
    df = df.withColumn("online_order", F.lit(False))

if "book_table" in df.columns:
    df = df.withColumn("book_table_cat", yn_unknown("book_table"))
    df = df.withColumn("book_table", to_bool_from_yn("book_table_cat"))
else:
    df = df.withColumn("book_table_cat", F.lit("Unknown"))
    df = df.withColumn("book_table", F.lit(False))

# --- Conditional integrity: offline -> null out delivery_time & distance ---
if "delivery_time" in df.columns:
    df = df.withColumn("delivery_time",
        F.when(F.col("online_order") == False, None).otherwise(F.col("delivery_time"))
    )
if "distance_km" in df.columns:
    df = df.withColumn("distance_km",
        F.when(F.col("online_order") == False, None).otherwise(F.col("distance_km"))
    )

# --- Delivery duration ---
if set(["order_time","delivery_time"]).issubset(df.columns):
    df = df.withColumn("delivery_minutes",
        F.when(F.col("order_time").isNotNull() & F.col("delivery_time").isNotNull(),
               (F.col("delivery_time").cast("long") - F.col("order_time").cast("long"))/60.0
        ).otherwise(None)
    )

# --- Final column selection ---
select_cols = [c for c in [
    "order_id","order_time","delivery_time","restaurant","cuisine",
    "total_amount","rating_num","location","city","distance_km",
    "online_order","book_table","online_order_cat","book_table_cat",
    "delivery_minutes","load_ts"
] if c in df.columns]

clean_df = df.select(*select_cols)

print("Clean preview:")
display(clean_df.limit(20))
print("Clean count:", clean_df.count())

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 12, Finished, Available, Finished, False)

Rows with negative total_amount (will be removed): 0
Clean preview:


SynapseWidget(Synapse.DataFrame, 19c8519e-d068-4ca2-813b-7e4b82cff6e4)

Clean count: 4000


In [12]:
if "rating_num" in clean_df.columns:
    med_row = clean_df.selectExpr("percentile_approx(rating_num, 0.5) AS med").collect()[0]
    rating_median = med_row["med"]
    print("rating_num median:", rating_median)
    clean_df = clean_df.withColumn(
        "rating_num",
        F.when(F.col("rating_num").isNull(), F.lit(rating_median)).otherwise(F.col("rating_num"))
    )

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 14, Finished, Available, Finished, False)

rating_num median: 3.4


In [14]:
# Save cleaned table
(clean_df
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(SILVER_TABLE)
)

# No need to drop first; this replaces if it exists
clean_df.createOrReplaceTempView(SILVER_VIEW)

print(f"Saved cleaned Delta table: {SILVER_TABLE}")

StatementMeta(, fe14ad22-9cd1-4ceb-b83f-1665b0c464ce, 16, Finished, Available, Finished, False)

Saved cleaned Delta table: silver_food_orders
